# Layer 2 Demo  -  Gap & Missingness Detection

This notebook is a thin runner around the installed `regvar` package  -  the actual pipeline logic
lives in `src/regvar/layer2/` (see `CONTRIBUTING.md` rule #5). Use this for interactive exploration;
the complete test suite is tested in `tests/layer2/`.

Full design rationale: `docs/framework_overview.md`.


In [ ]:
# Setup environment if running in Colab or fresh clone
import os
if not os.path.exists("pyproject.toml"):
    !git clone -q https://github.com/utk1017/regulatory-variant-prioritization.git repo
    %cd repo

!pip install -q -e .


In [ ]:
from regvar.layer2.pipeline import run_layer2

# Execute Layer 2 Gap & Missingness Detection
availability_df, triggers_df, agreement_df = run_layer2("configs/hbb.yaml")

if availability_df is not None and not availability_df.empty:
    print(f"Modality Availability: {len(availability_df)} candidate variants")
    display(availability_df.head(10))


## Inspect AlphaGenome Triggers and Agreement Checks

- **AlphaGenome Trigger Queue**: Jobs queued for missing completable modalities (`eqtl`, `mpra`, `crispr`).
- **Agreement Check Queue**: Observed experimental measurements on completable modalities reserved for benchmarking.

In [ ]:
if triggers_df is not None and not triggers_df.empty:
    print(f"AlphaGenome Triggers ({len(triggers_df)} jobs):")
    display(triggers_df.head(10))

if agreement_df is not None and not agreement_df.empty:
    print(f"Agreement Checks ({len(agreement_df)} benchmarks):")
    display(agreement_df.head(10))


## Evidence Stratification (Tier Distribution)

Variants are stratified into evidence tiers for downstream calibrated prior estimation:
- **Tier 1 (Evidence-Rich)**: $\ge 2$ experimental regulatory modalities tested.
- **Tier 2 (Moderate Evidence)**: Exactly 1 experimental regulatory modality tested.
- **Tier 3 (Evidence-Sparse)**: 0 experimental regulatory modalities tested.

In [ ]:
if availability_df is not None and not availability_df.empty:
    display(availability_df["evidence_tier"].value_counts().rename("variant_count").to_frame())


## Inspect Provenance Manifest

Every run writes `layer2_manifest.json` capturing effective config, data inputs, hashes,
warnings, and trigger breakdown.

In [ ]:
import json
for path in ["data/outputs/HBB/layer2_manifest.json", "data/outputs/HBB/synthetic_demo/layer2_manifest.json"]:
    if os.path.exists(path):
        with open(path) as f:
            manifest = json.load(f)
        print(f"Loaded manifest from {path}:")
        print(json.dumps(manifest, indent=2))
        break
